# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print('Total revenue:', total_revenue)
print('Total units:', total_units)

Total revenue: 8520.0
Total units: 783


The 400 orders generated $8,520 in total revenue and included 783 units sold.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [15]:
by_category = (
    df.groupby('category')['revenue']
      .sum()
      .sort_values(ascending=False)
      .to_frame()
)

by_category['share_of_total'] = (
    by_category['revenue'] / df['revenue'].sum() * 100
)

by_category

,revenue,share_of_total
category,,
Food,4293.0,50.387324
Merch,1771.5,20.792254
Drink,1554.0,18.239437
RainGear,901.5,10.580986


Food generated the most revenue at $4,293, or 50.4% of total revenue, followed by Merch, Drink, and RainGear.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [8]:
vendor_summary = (
    df.groupby('vendor_id')
      .agg(
          avg_order_revenue=('revenue', 'mean'),
          order_count=('revenue', 'size')
      )
      .sort_values('avg_order_revenue', ascending=False))

vendor_summary

,avg_order_revenue,order_count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


V-01 had the highest average order revenue at $22.60 per order across 94 orders, so its high average is based on a reasonably large number of transactions.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [9]:
merch_share = (
    df.loc[df['category'] == 'Merch', 'revenue'].sum()
    / df['revenue'].sum()
    * 100)

print(f"{merch_share:.1f}%")

20.8%


Merch accounted for 20.8% of total revenue, meaning about one-fifth of all revenue came from Merch sales.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [20]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

df = df[['vendor_id', 'category', 'qty', 'price', 'revenue']].copy()

rows_before = len(df)
revenue_before = df['revenue'].sum()

joined = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)

print('Rows before:', rows_before)
print('Rows after:', len(joined))
print('Revenue before:', revenue_before)
print('Revenue after:', joined['revenue'].sum())

assert len(joined) == rows_before
assert joined['revenue'].sum() == revenue_before

missing_vendor = joined.loc[
    joined['vendor_name'].isna(),
    'vendor_id'
].unique()

print('Missing vendor:', missing_vendor)

joined['vendor_name'] = joined['vendor_name'].fillna('Unknown Vendor')

joined.head()

Rows before: 400
Rows after: 400
Revenue before: 8520.0
Revenue after: 8520.0
Missing vendor: ['V-18']


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unknown Vendor
2,V-18,Drink,3,4.5,13.5,Unknown Vendor
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unknown Vendor


The left join preserved all 400 rows and $8,520 in revenue. V-18 was unmatched, so I kept its orders and labeled it “Unknown Vendor.”

The unmatched vendor was V-18, because it did not appear in the vendor lookup table.
I kept its orders and labeled the vendor as “Unknown Vendor” so no valid revenue or transactions were lost.


### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [14]:
vendor_category_pivot = pd.pivot_table(
    df,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

vendor_category_pivot

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown Vendor,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


The pivot table shows revenue by vendor and category; Food was the largest category at \$4,293, and Unknown Vendorhad  the highest vendor total at \$2,349.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [21]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'

print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a)I would tell the vendors to prepare more heavily for Food demand, since Food generated \$4,293, or 50.4% of total revenue, making it the largest category. I would also pay attention to V-18, which generated \$2,349 in total revenue, the highest among the vendors, and make sure its most popular items are well stocked for the next game. Merch and Drink still contribute meaningful revenue, so vendors should keep those available, but inventory and staffing should be weighted more toward Food.

b)The least trustworthy answer is the vendor-name analysis because V-18 was missing from the vendor lookup table and had to be labeled as “Unknown Vendor.” Although its orders and revenue were preserved, not knowing the actual vendor name makes that part of the report less complete and less useful for giving a specific vendor recommendation.